In [ ]:
import datetime as dt
import os

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from mc_postgres_db.models import Asset, Provider, ProviderAssetMarket
from sqlalchemy import create_engine, select
from sqlalchemy.orm import Session, aliased
from statsmodels.tsa.stattools import adfuller, coint
from tqdm.notebook import tqdm

load_dotenv()

POSTGRES_URL = os.getenv("POSTGRES_URL")

engine = create_engine(POSTGRES_URL)

In [ ]:
with Session(engine) as session:
    stmt = select(Provider).where(Provider.name == "Kraken")
    kraken = session.execute(stmt).scalar_one()
    display(kraken)

In [ ]:
with Session(engine) as session:
    stmt = select(Asset).where(Asset.name == "USD")
    usd_asset = session.execute(stmt).scalar_one()
    display(usd_asset)

In [ ]:
id_to_asset_dict = {}
with Session(engine) as session:
    stmt = select(Asset)
    for asset in session.execute(stmt).scalars():
        id_to_asset_dict[asset.id] = asset

In [ ]:
start = dt.datetime(2024, 1, 21, 0, 0, 0)
end = start + dt.timedelta(days=7)

from_asset = aliased(Asset)
to_asset = aliased(Asset)

stmt = (
    select(
        ProviderAssetMarket.timestamp,
        ProviderAssetMarket.from_asset_id,
        ProviderAssetMarket.to_asset_id,
        from_asset.name.label("from_asset_name"),
        to_asset.name.label("to_asset_name"),
        ProviderAssetMarket.close,
        ProviderAssetMarket.volume,
    )
    .where(ProviderAssetMarket.timestamp >= start, ProviderAssetMarket.timestamp <= end)
    .join(from_asset, ProviderAssetMarket.from_asset_id == from_asset.id)
    .join(to_asset, ProviderAssetMarket.to_asset_id == to_asset.id)
    .where(
        (ProviderAssetMarket.provider_id == kraken.id) & (from_asset.id == usd_asset.id)
    )
)
df = pd.read_sql(stmt, engine)
df["volume_usd"] = df["volume"] * df["close"]

df

In [ ]:
asset_volume_df = (
    df.groupby(by=["from_asset_id", "to_asset_id"])["volume_usd"]
    .sum()
    .sort_values(ascending=False)
    .reset_index(name="volume_usd_sum")
)
asset_volume_df

In [ ]:
# Find pairs with at least minimum volume.
minimum_volume_usd = pow(10, 6)
keep_asset_pairs_df = (
    asset_volume_df.loc[asset_volume_df["volume_usd_sum"] >= minimum_volume_usd][
        ["from_asset_id", "to_asset_id"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)
keep_asset_pairs_df["keep"] = True

# Drop pairs with less than minimum volume.
df = df.merge(keep_asset_pairs_df, on=["from_asset_id", "to_asset_id"], how="left")

# Print out the dropped pairs.
dropped_asset_pairs_df = (
    asset_volume_df.loc[asset_volume_df["volume_usd_sum"] < minimum_volume_usd][
        ["from_asset_id", "to_asset_id"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)
dropped_asset_pairs_df["dropped"] = True
dropped_to_assets = dropped_asset_pairs_df["to_asset_id"].drop_duplicates().to_list()
print(
    f"Dropped {len(dropped_to_assets)} to_assets with less than {minimum_volume_usd} volume: {[id_to_asset_dict[asset_id].name for asset_id in dropped_to_assets]}"
)

# Drop pairs with less than minimum volume.
df = df.loc[df["keep"]].reset_index(drop=True)
df = df.drop(columns=["keep"])

In [ ]:
print(
    f"Unique to_asset_name: {df['to_asset_name'].sort_values().drop_duplicates().to_list()}"
)
print(
    f"Unique from_asset_name: {df['from_asset_name'].sort_values().drop_duplicates().to_list()}"
)

In [ ]:
from itertools import combinations

trading_pairs = (
    df[["from_asset_id", "to_asset_id"]]
    .drop_duplicates()
    .apply(
        lambda x: (x["from_asset_id"], x["to_asset_id"]),
        axis=1,
    )
    .to_list()
)

pair_combinations = list(combinations(trading_pairs, 2))
num_combinations = len(pair_combinations)

print(f"Number of combinations: {num_combinations}")

In [ ]:
def align_series(
    df_1: pd.DataFrame, df_2: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Check for required columns.
    if "timestamp" not in df_1.columns or "timestamp" not in df_2.columns:
        raise ValueError("Timestamp column is required.")
    if "close" not in df_1.columns or "close" not in df_2.columns:
        raise ValueError("Close column is required.")

    # If either series is empty, return the empty series.
    if df_1.empty:
        return df_1, df_1
    if df_2.empty:
        return df_2, df_2

    # Get the start and end dates of the series.
    start = max(
        df_1["timestamp"].min(),
        df_2["timestamp"].min(),
    )
    end = min(
        df_1["timestamp"].max(),
        df_2["timestamp"].max(),
    )

    # Forward fill the series and join to a time-frame for missing values.
    time_df = pd.DataFrame(
        {"timestamp": pd.Series(np.arange(start, end, np.timedelta64(1, "1m")))}
    )
    df_1 = time_df.merge(df_1, on="timestamp", how="left").reset_index(drop=True)
    df_2 = time_df.merge(df_2, on="timestamp", how="left").reset_index(drop=True)
    df_1 = df_1.ffill()
    df_2 = df_2.ffill()

    # Drop rows with missing values.
    df_1 = df_1.dropna()
    df_2 = df_2.dropna()

    return df_1, df_2


# Log normalization (for positive values)
def log_normalize(series):
    return np.log(series - np.min(series) + 1)


def min_max_normalize(series):
    return (series - np.min(series)) / (np.max(series) - np.min(series))

In [ ]:
# Do data filtering ahead of time.
pair_df_dict = {}
for pair_1, pair_2 in pair_combinations:
    if pair_1 not in pair_df_dict:
        pair_df_dict[pair_1] = df.loc[
            (df["from_asset_id"] == pair_1[0]) & (df["to_asset_id"] == pair_1[1])
        ].reset_index(drop=True)
        pair_df_dict[pair_1] = pair_df_dict[pair_1].set_index("timestamp")
        pair_df_dict[pair_1] = (
            pair_df_dict[pair_1].resample("15min").last().reset_index()
        )
    if pair_2 not in pair_df_dict:
        pair_df_dict[pair_2] = df.loc[
            (df["from_asset_id"] == pair_2[0]) & (df["to_asset_id"] == pair_2[1])
        ].reset_index(drop=True)
        pair_df_dict[pair_2] = pair_df_dict[pair_2].set_index("timestamp")
        pair_df_dict[pair_2] = (
            pair_df_dict[pair_2].resample("15min").last().reset_index()
        )

In [ ]:
# Set the p-value cutoffs.
pvalue_cutoff = 0.01
cpvalue_cutoff = 0.01

# Calculate the candidate pairs.
candidate_pairs = []
for pair_1, pair_2 in tqdm(pair_combinations):
    # Get the assets.
    pair_1_from_asset = id_to_asset_dict[pair_1[0]]
    pair_1_to_asset = id_to_asset_dict[pair_1[1]]
    pair_2_from_asset = id_to_asset_dict[pair_2[0]]
    pair_2_to_asset = id_to_asset_dict[pair_2[1]]

    # Normalize the series.
    data_1, data_2 = align_series(pair_df_dict[pair_1], pair_df_dict[pair_2])

    # If the series contains any NaNs or Infinities, skip.
    if (
        data_1["close"].isna().any()
        or data_2["close"].isna().any()
        or data_1["close"].isin([np.inf, -np.inf]).any()
        or data_2["close"].isin([np.inf, -np.inf]).any()
    ):
        print(
            f"Skipping pair {pair_1_to_asset.name}-{pair_1_from_asset.name} or {pair_2_to_asset.name}-{pair_2_from_asset.name} because it contains NaNs or Infinities."
        )
        continue

    # Calculate the spread.
    spread = data_1["close"] - data_2["close"]

    # If the series contains any NaNs or Infinities, skip.
    if spread.isna().any() or spread.isin([np.inf, -np.inf]).any():
        print(
            f"Skipping pair {pair_1_to_asset.name}-{pair_1_from_asset.name} or {pair_2_to_asset.name}-{pair_2_from_asset.name} because the spread contains NaNs or Infinities."
        )
        continue

    # Calculate the ADF p-value.
    _, adf_p_value, _, _, _, _ = adfuller(spread)

    # Calculate the cointegration p-value.
    _, coint_p_value, _ = coint(
        log_normalize(data_1["close"]), log_normalize(data_2["close"])
    )

    # Add the candidate pair to the list.
    candidate_pairs.append(
        {
            "to_asset_1": pair_1_to_asset,
            "from_asset_1": pair_1_from_asset,
            "to_asset_2": pair_2_to_asset,
            "from_asset_2": pair_2_from_asset,
            "spread_adf_pvalue": adf_p_value,
            "cointegration_pvalue": coint_p_value,
            "is_spread_adf_stationary": adf_p_value < pvalue_cutoff,
            "is_cointegrated": coint_p_value < cpvalue_cutoff,
        }
    )

candidate_pairs_df = pd.DataFrame(candidate_pairs)
candidate_pairs_df

In [ ]:
picked_pairs = candidate_pairs_df.loc[
    candidate_pairs_df["is_spread_adf_stationary"]
    & candidate_pairs_df["is_cointegrated"]
].reset_index(drop=True)

print(
    f"Found {len(picked_pairs)} pairs that have a stationary spread and are cointegrated."
)

picked_pairs

In [ ]:
# Set the number of rows and columns.
M = 2
N = min(int(len(picked_pairs) / M), 2)

# Create the figure and axes.
fig, axes = plt.subplots(
    N, M, figsize=(M * 5, N * 5)
)  # Fixed figsize and created axes array

for i, picked_pair in enumerate(picked_pairs.head(N * M).itertuples()):
    row = i // M  # Calculate row index
    col = i % M  # Calculate column index

    try:
        data_1, data_2 = align_series(
            pair_df_dict[(picked_pair.from_asset_1.id, picked_pair.to_asset_1.id)],
            pair_df_dict[(picked_pair.from_asset_2.id, picked_pair.to_asset_2.id)],
        )

        spread = min_max_normalize(data_1["close"]) - min_max_normalize(data_2["close"])

        # Calculate the mean and std of the spread.
        mean = spread.mean()
        std = spread.std()

        # Get arrays of the mean + 2 * std and mean - 2 * std.
        mean_array = np.ones(len(spread)) * mean
        std_array = np.ones(len(spread)) * std
        mean_plus_2_std_array = mean_array + 2 * std_array
        mean_minus_2_std_array = mean_array - 2 * std_array

        # Access the correct subplot using 2D indexing, plot the mean and std lines.
        axes[row, col].plot(data_1["timestamp"], spread.values, linewidth=1)
        axes[row, col].plot(
            data_1["timestamp"], mean_array, linewidth=1, color="red", linestyle="--"
        )
        axes[row, col].plot(
            data_1["timestamp"],
            mean_plus_2_std_array,
            linewidth=1,
            color="green",
            linestyle="--",
        )
        axes[row, col].plot(
            data_1["timestamp"],
            mean_minus_2_std_array,
            linewidth=1,
            color="green",
            linestyle="--",
        )
        axes[row, col].xaxis.set_major_locator(mdates.AutoDateLocator())
        axes[row, col].tick_params(axis="x", rotation=45, labelsize=8)
        axes[row, col].set_title(
            f"{picked_pair.to_asset_1.name}/{picked_pair.from_asset_1.name} vs {picked_pair.to_asset_2.name}/{picked_pair.from_asset_2.name}"
        )
        axes[row, col].legend(
            ["Spread (Min-Max Normalized)", "Mean", "Mean + 2 Std", "Mean - 2 Std"]
        )
        axes[row, col].set_ylim(mean - 10 * std, mean + 10 * std)
        axes[row, col].set_xlabel("Time")
        axes[row, col].set_ylabel("Spread (Min-Max Normalized)")
        axes[row, col].grid(True, alpha=0.3)

    except Exception as e:
        print(f"Error processing pair {i}: {e}")

fig.tight_layout()
plt.show()

In [ ]:
next_period_start = end
next_period_end = next_period_start + dt.timedelta(days=7)

stmt = (
    select(
        ProviderAssetMarket.timestamp,
        ProviderAssetMarket.from_asset_id,
        ProviderAssetMarket.to_asset_id,
        from_asset.name.label("from_asset_name"),
        to_asset.name.label("to_asset_name"),
        ProviderAssetMarket.close,
        ProviderAssetMarket.volume,
    )
    .where(
        ProviderAssetMarket.timestamp >= next_period_start,
        ProviderAssetMarket.timestamp <= next_period_end,
    )
    .join(from_asset, ProviderAssetMarket.from_asset_id == from_asset.id)
    .join(to_asset, ProviderAssetMarket.to_asset_id == to_asset.id)
    .where(ProviderAssetMarket.provider_id == kraken.id)
)
df_test = pd.read_sql(stmt, engine)
df_test = pd.concat([df, df_test])
df_test = df_test.sort_values(by=["timestamp"])
df_test = df_test[["timestamp", "from_asset_id", "to_asset_id", "close"]]

df_test

In [ ]:
# Do data filtering ahead of time.
df_pairs = pd.DataFrame(
    columns=[
        "timestamp",
        "from_asset_id_1",
        "to_asset_id_1",
        "from_asset_id_2",
        "to_asset_id_2",
        "close_1",
        "close_2",
        "log_close_1",
        "log_close_2",
        "spread",
    ]
)
for pair in picked_pairs.itertuples():
    # Get the from and to asset ids and names.
    from_asset_1 = pair.from_asset_1
    to_asset_1 = pair.to_asset_1
    from_asset_2 = pair.from_asset_2
    to_asset_2 = pair.to_asset_2

    # Get the data for the pair.
    data_1 = df_test.loc[
        (df_test["from_asset_id"] == from_asset_1.id)
        & (df_test["to_asset_id"] == to_asset_1.id)
    ].reset_index(drop=True)

    # Get the data for the pair.
    data_2 = df_test.loc[
        (df_test["from_asset_id"] == from_asset_2.id)
        & (df_test["to_asset_id"] == to_asset_2.id)
    ].reset_index(drop=True)

    # Align the series.
    data_1, data_2 = align_series(data_1, data_2)

    # If the series contains any NaNs or Infinities, skip.
    if (
        data_1["close"].isna().any()
        or data_2["close"].isna().any()
        or data_1["close"].isin([np.inf, -np.inf]).any()
        or data_2["close"].isin([np.inf, -np.inf]).any()
    ):
        print(
            f"Skipping pair {pair_1_to_asset.name}-{pair_1_from_asset.name} or {pair_2_to_asset.name}-{pair_2_from_asset.name} because it contains NaNs or Infinities."
        )
        continue

    # If the two series are not the same length, skip.
    if len(data_1) != len(data_2):
        print(
            f"Skipping pair {pair_1_to_asset.name}-{pair_1_from_asset.name} or {pair_2_to_asset.name}-{pair_2_from_asset.name} because the series are not the same length."
        )
        continue

    # Merge the series.
    data_combined = pd.merge(
        data_1, data_2, on="timestamp", how="inner", suffixes=("_1", "_2")
    )
    data_combined["log_close_1"] = log_normalize(data_combined["close_1"])
    data_combined["log_close_2"] = log_normalize(data_combined["close_2"])
    data_combined["spread"] = (
        data_combined["log_close_1"] - data_combined["log_close_2"]
    )

    # Add the pair to the dataframe.
    df_pairs = pd.concat([df_pairs, data_combined])

display(
    df_pairs.sort_values(
        by=["from_asset_id_1", "to_asset_id_1", "from_asset_id_2", "to_asset_id_2"]
    )
)

In [ ]:
# Calculate the rolling mean and std of the spread.
window = 7 * 24 * 60
rolling_mean_spread_df = (
    df_pairs.sort_values(by="timestamp")
    .set_index("timestamp")
    .groupby(["from_asset_id_1", "to_asset_id_1", "from_asset_id_2", "to_asset_id_2"])[
        "spread"
    ]
    .rolling(window=window, min_periods=window)
    .mean()
    .reset_index(name="rolling_mean_spread")
)
rolling_mean_std_df = (
    df_pairs.sort_values(by="timestamp")
    .set_index("timestamp")
    .groupby(["from_asset_id_1", "to_asset_id_1", "from_asset_id_2", "to_asset_id_2"])[
        "spread"
    ]
    .rolling(window=window, min_periods=window)
    .std()
    .reset_index(name="rolling_mean_std")
)
df_pairs = df_pairs.merge(
    rolling_mean_spread_df,
    on=[
        "timestamp",
        "from_asset_id_1",
        "to_asset_id_1",
        "from_asset_id_2",
        "to_asset_id_2",
    ],
    how="left",
)
df_pairs = df_pairs.merge(
    rolling_mean_std_df,
    on=[
        "timestamp",
        "from_asset_id_1",
        "to_asset_id_1",
        "from_asset_id_2",
        "to_asset_id_2",
    ],
    how="left",
)
df_rolling_means = df_pairs.dropna(
    subset=["rolling_mean_spread", "rolling_mean_std"]
).reset_index(drop=True)
display(df_rolling_means)

In [ ]:
df_rolling_means["rolling_z_score"] = (
    df_rolling_means["spread"] - df_rolling_means["rolling_mean_spread"]
) / df_rolling_means["rolling_mean_std"]
df_rolling_means["signal_enter"] = df_rolling_means["rolling_z_score"].abs() > 3
df_rolling_means["signal_exit"] = df_rolling_means["rolling_z_score"].abs() < 1
df_rolling_means["enter_long_spread"] = df_rolling_means["signal_enter"] & (
    df_rolling_means["rolling_z_score"] > 0
)
df_rolling_means["enter_short_spread"] = df_rolling_means["signal_enter"] & (
    df_rolling_means["rolling_z_score"] < 0
)
df_rolling_means["exit_long_spread"] = df_rolling_means["signal_exit"] & (
    df_rolling_means["rolling_z_score"] < 0
)
df_rolling_means["exit_short_spread"] = df_rolling_means["signal_exit"] & (
    df_rolling_means["rolling_z_score"] > 0
)
df_rolling_means

In [ ]:
df_rolling_means.loc[df_rolling_means["signal"]]

In [ ]:
df_pairs["signal"] = df_pairs["rolling_z_score"].abs() > 3
df_pairs["long_spread"] = df_pairs["signal"] & (
    np.sign(df_pairs["rolling_z_score"]) > 0
)
df_pairs["short_spread"] = ~df_pairs["long_spread"]
df_pairs